# Explore here

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
url = "https://raw.githubusercontent.com/4GeeksAcademy/naive-bayes-project-tutorial/main/playstore_reviews.csv"
df = pd.read_csv(url)

# Eliminamos package_name: no aporta señal sobre el sentimiento del comentario,
# solo identifica la app de origen, lo que podría hacer que el modelo memorice
# la app en vez de aprender del contenido del texto
df = df.drop("package_name", axis=1)

df.head()

,review,polarity
0,privacy at least put some option appear offli...,0
1,"messenger issues ever since the last update, ...",0
2,profile any time my wife or anybody has more ...,0
3,the new features suck for those of us who don...,0
4,forced reload on uploading pic on replying co...,0


In [5]:
# Preprocesado de texto: quitar espacios y pasar a minúsculas
df["review"] = df["review"].str.strip().str.lower()

df.head()

,review,polarity
0,privacy at least put some option appear offlin...,0
1,"messenger issues ever since the last update, i...",0
2,profile any time my wife or anybody has more t...,0
3,the new features suck for those of us who don'...,0
4,forced reload on uploading pic on replying com...,0


In [6]:
from sklearn.model_selection import train_test_split

X = df["review"]
y = df["polarity"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train.shape, X_test.shape

((712,), (179,))

In [8]:
from sklearn.feature_extraction.text import CountVectorizer

vec_model = CountVectorizer(stop_words="english")
X_train_vec = vec_model.fit_transform(X_train).toarray()
X_test_vec = vec_model.transform(X_test).toarray()

In [9]:
from sklearn.naive_bayes import MultinomialNB, GaussianNB, BernoulliNB
from sklearn.metrics import accuracy_score, f1_score

modelos = {
    "MultinomialNB": MultinomialNB(),
    "GaussianNB": GaussianNB(),
    "BernoulliNB": BernoulliNB()
}

resultados = {}
for nombre, modelo in modelos.items():
    modelo.fit(X_train_vec, y_train)
    preds = modelo.predict(X_test_vec)
    resultados[nombre] = {
        "accuracy": accuracy_score(y_test, preds),
        "f1": f1_score(y_test, preds)
    }

for nombre, metricas in resultados.items():
    print(f"{nombre}: accuracy={metricas['accuracy']:.4f}, f1={metricas['f1']:.4f}")

MultinomialNB: accuracy=0.8156, f1=0.6598
GaussianNB: accuracy=0.8045, f1=0.6535
BernoulliNB: accuracy=0.7709, f1=0.5060


- MultinomialNB gana, como esperábamos, porque es el único que usa la información completa de los conteos (no solo presencia/ausencia, ni asume una distribución continua que no es real aquí).
- GaussianNB queda muy cerca, sin embargo conceltualmente, esta distribución se usa principalmente cuando los predictores son varibles numéricas.
- BernoulliNB es el que peor rinde, con una caída en F1 (0.506). Esto pasa porque hace un conteo de las palabras y no de cuantas veces aparecen, es decir, no identifica bien las veces que aparecen palabras, por ejemplo, 1 Terrible y 5 Terrible sería lo mismo, y en un dataset con desbalance de clases (65/35) esa pérdida de afecta más al F1 que al accuracy.

La idea es tomar la mejor opción de Naive Bayes (MultinomialNB) como referencia, y ver si un modelo más complejo como RandomForestClassifier puede superarla, usando las mismas matrices de conteo de palabras como entrada.

In [10]:
from sklearn.ensemble import RandomForestClassifier

# Referencia: MultinomialNB (paso 3)
# accuracy = 0.8156, f1 = 0.6598

# Random Forest con parámetros por defecto
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train_vec, y_train)
preds_rf = rf.predict(X_test_vec)

print(f"RandomForest (default): accuracy={accuracy_score(y_test, preds_rf):.4f}, f1={f1_score(y_test, preds_rf):.4f}")

RandomForest (default): accuracy=0.7989, f1=0.6842


Es un resultado mixto: Random Forest tiene menos accuracy pero más F1 que MultinomialNB. Dado que tenemos desbalance de clases (65% negativas / 35% positivas), el F1 es una métrica más honesta aquí, ya que un modelo puede tener buena accuracy simplemente acertando mucho en la clase mayoritaria, mientras que F1 exige balancear precision y recall en la clase positiva.

Como esto no es una mejora clara y contundente en ambas métricas, se intentará optimizar los hiperparámetros de Random Forest.

In [11]:
from sklearn.ensemble import RandomForestClassifier

configs = [
    ("default", dict()),
    ("n_estimators=300", dict(n_estimators=300)),
    ("max_depth=20", dict(max_depth=20)),
    ("class_weight=balanced", dict(class_weight="balanced")),
    ("n_estimators=300 + class_weight=balanced", dict(n_estimators=300, class_weight="balanced")),
]

for nombre, params in configs:
    rf = RandomForestClassifier(random_state=42, **params)
    rf.fit(X_train_vec, y_train)
    preds = rf.predict(X_test_vec)
    print(f"{nombre}: accuracy={accuracy_score(y_test, preds):.4f}, f1={f1_score(y_test, preds):.4f}")

default: accuracy=0.7989, f1=0.6842
n_estimators=300: accuracy=0.8156, f1=0.7080
max_depth=20: accuracy=0.7207, f1=0.3056
class_weight=balanced: accuracy=0.7374, f1=0.6412
n_estimators=300 + class_weight=balanced: accuracy=0.7486, f1=0.6565


n_estimators solo, es el mejor: más árboles reduce la varianza del ensamble sin cambiar nada más, y esto solito nos da la mejor combinación de accuracy y F1.

|Modelo	|Accuracy |F1 |
|-------|-------|-------|
|MultinomialNB |0.8156 |0.6598 |
|RandomForest (n_estimators=300)| 0.8156 | 0.7080|

Conclusión: Random Forest con n_estimators=300 iguala la accuracy de MultinomialNB y mejora claramente el F1 (0.708 vs 0.660), que es la métrica que más nos importa dado el desbalance de clases.

In [17]:
import pickle

# Modelo final: RandomForest con n_estimators=300
final_model = RandomForestClassifier(n_estimators=300, random_state=42)
final_model.fit(X_train_vec, y_train)

# Guardamos el modelo
with open("models/random_forest_sentiment_n300.pkl", "wb") as f:
    pickle.dump(final_model, f)

# También guardamos el CountVectorizer: sin él no podremos transformar
# texto nuevo de la misma forma que se entrenó el modelo.
with open("models/count_vectorizer.pkl", "wb") as f:
    pickle.dump(vec_model, f)

In [15]:
import os

print("Directorio actual del notebook:", os.getcwd())
print("Contenido de ese directorio:", os.listdir())

Directorio actual del notebook: /workspaces/machine-learning-project-python-naive-bayes/src
Contenido de ese directorio: ['explore.ipynb', 'app.py', 'utils.py']


In [ ]:
import os
os.makedirs("models", exist_ok=True)

**Observaciones**
El desbalance entre dimensiones (3310) y ejemplos (712), los modelos lineales (Logistic Regression, LinearSVC) tienen más probabilidad de generalizar bien que modelos basados en árboles adicionales. Vamos a probar ambos y comparar contra nuestra tabla de resultados.

In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

modelos_alt = {
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
    "LinearSVC": LinearSVC(random_state=42)
}

for nombre, modelo in modelos_alt.items():
    modelo.fit(X_train_vec, y_train)
    preds = modelo.predict(X_test_vec)
    print(f"{nombre}: accuracy={accuracy_score(y_test, preds):.4f}, f1={f1_score(y_test, preds):.4f}")

LogisticRegression: accuracy=0.8324, f1=0.7414
LinearSVC: accuracy=0.8324, f1=0.7458


**Conclusiones**

|Modelo	|Accuracy |F1 |
|-------|-------|-------|
|MultinomialNB (default) |0.8156 |0.6598 |
|RandomForest (n_estimators=300)| 0.8156 | 0.7080|
|LogisticRegression (default) |0.8324 |0.7414 |
|LinearSVC (default) |0.8324 |0.7458 |

Los dos modelos lineales superan a Naive Bayes como a Random Forest, en accuracy y en F1, y sin siquiera necesitar optimización de hiperparámetros (valores por defecto) ya rinden mejor, esto puede pasar porque los modelos de decisión lineal identifican mejor que posiblemente algunas palabras clave hagan la diferencia en la aceptación o el rechazo.

In [20]:
# Mejor modelo en carpeta Models
from sklearn.svm import LinearSVC
import pickle, os

# Modelo final: LinearSVC
final_model = LinearSVC(random_state=42)
final_model.fit(X_train_vec, y_train)

os.makedirs("models", exist_ok=True)

with open("models/linearsvc_sentiment.pkl", "wb") as f:
    pickle.dump(final_model, f)

with open("models/count_vectorizer.pkl", "wb") as f:
    pickle.dump(vec_model, f)